# Notebook 2 — Data Pipeline: DataLoader, Augmentation, Format Conversion

**Bài tập lớn số 2 · CO5085 · HCMUT 2025-2026**

## Mục tiêu
1. Hiểu tại sao hai mô hình cần hai format dữ liệu khác nhau
2. Demo VOCDetectionDataset và custom collate_fn
3. Visualize augmentation cho ảnh + bounding boxes
4. Chuyển đổi VOC XML → YOLO txt format

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.data import (VOCDetectionDataset, get_train_transforms, get_val_transforms,
                      collate_fn, get_frcnn_loaders, prepare_yolo_dataset, VOC_CLASSES)
from src.utils import visualize_detections, denormalize_imagenet

## 1. Tại sao cần hai format khác nhau?

| Mô hình | Format ảnh | Format annotation |
|---------|-----------|------------------|
| **Faster R-CNN** | `list[Tensor[C,H,W]]` in [0,1] | `list[dict]` với `boxes` [N,4] tuyệt đối |
| **YOLOv8** | File ảnh JPEG | File `.txt` với `class xc yc w h` normalized |

Faster R-CNN (torchvision) nhận list vì ảnh có kích thước khác nhau — **không thể stack**.
YOLOv8 (ultralytics) có pipeline riêng đọc từ file.

In [ ]:
# Demo VOCDetectionDataset
ds = VOCDetectionDataset('../data/voc', year='2012', image_set='val',
                          transforms=get_val_transforms())

img, target = ds[0]
print("Image shape:", img.shape)
print("Boxes:", target['boxes'])
print("Labels:", target['labels'])
print("Label names:", [VOC_CLASSES[l-1] for l in target['labels'].tolist()])

## 2. Custom collate_fn

In [ ]:
from torch.utils.data import DataLoader

# Thử tạo DataLoader với collate_fn tùy chỉnh
loader = DataLoader(ds, batch_size=2, collate_fn=collate_fn)
imgs, targets = next(iter(loader))

print(f"Type imgs: {type(imgs)} | len: {len(imgs)}")
print(f"Type targets: {type(targets)} | len: {len(targets)}")
print(f"Image 0 shape: {imgs[0].shape}")
print(f"Target 0 boxes shape: {targets[0]['boxes'].shape}")
print()
print("Tại sao không dùng torch.stack?")
print("→ Mỗi ảnh VOC có kích thước khác nhau!")
print(f"  img[0]: {imgs[0].shape[1:]}")
print(f"  img[1]: {imgs[1].shape[1:]}")

## 3. Augmentation Visualization

In [ ]:
from torchvision import transforms

# Augmentation áp dụng cho training
train_tf = get_train_transforms()
val_tf = get_val_transforms()

ds_train = VOCDetectionDataset('../data/voc', year='2012', image_set='val',
                                transforms=None)

img_pil, target = ds_train[5]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Ảnh gốc
import numpy as np
axes[0].imshow(img_pil)
axes[0].set_title('Ảnh gốc (PIL)')
axes[0].axis('off')

# Val transform (chỉ normalize)
img_val = val_tf(img_pil)
axes[1].imshow(denormalize_imagenet(img_val))
axes[1].set_title('Val transform\n(ToTensor + Normalize)')
axes[1].axis('off')

# Train transform (augmentation + normalize)
import random; random.seed(42); torch.manual_seed(42)
img_aug = train_tf(img_pil)
axes[2].imshow(denormalize_imagenet(img_aug))
axes[2].set_title('Train transform\n(Flip + ColorJitter + Normalize)')
axes[2].axis('off')

plt.suptitle('So sánh Val vs Train Transforms', fontsize=12)
plt.tight_layout()
plt.savefig('../results/plots/augmentation_demo.png', dpi=120)
plt.show()
print("Chú ý: Faster R-CNN xử lý augmentation bounding box separately.")

## 4. YOLO Format Conversion

In [ ]:
# Chuyển VOC XML → YOLO txt
yaml_path = prepare_yolo_dataset(
    voc_root='../data/voc',
    output_dir='../data/voc_yolo',
    splits=['train', 'val'],
)
print(f"\nYAML config: {yaml_path}")

In [ ]:
# Xem nội dung file voc.yaml
print(open(yaml_path).read())

In [ ]:
# Xem ví dụ một file label YOLO
from pathlib import Path
lbl_dir = Path('../data/voc_yolo/labels/val')
sample_lbl = next(lbl_dir.glob('*.txt'))
print(f"File: {sample_lbl.name}")
print("Nội dung (class xc yc w h - tất cả normalized [0,1]):")
print(sample_lbl.read_text()[:500])
print()
print("So sánh với VOC XML (absolute pixels): xmin, ymin, xmax, ymax")

## Tóm tắt Data Pipeline

```
Pascal VOC XML (absolute xyxy)
        │
        ├─→ VOCDetectionDataset ──→ list[(img_tensor, target_dict)] ──→ Faster R-CNN
        │
        └─→ prepare_yolo_dataset ──→ images/ + labels/*.txt + voc.yaml ──→ YOLOv8
```

**ImageNet Normalization** (mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]):
- Cả Faster R-CNN lẫn YOLOv8 đều dùng backbone pretrained trên ImageNet
- Normalize giúp input distribution match với distribution lúc pretrain